## Prompt Testing Notebook

The purpose of this notebook is to rapidly iterate prompt techniques for Gemma and Molmo to get them in line with the caption format of COCO. 

This will allow for accurate, useful metrics on model comparisons and ablation tests.

In [ ]:
#imports 

import os
import json
import random
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM, AutoProcessor

# Metrics
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

### Inputs!

In [ ]:
MODEL_NAME = "google/gemma-4-E4B-it" # Options: "google/gemma-4-E4B-it", "allenai/Molmo2-4B"
DATASET_PATH = "coco-dataset/val2017"
ANNOTATIONS_PATH = "coco-dataset/annotations/captions_val2017.json"

# Iteration Variables
PROMPT = "Describe this image in one brief sentence. Do not use conversational language."
MAX_NEW_TOKENS = 2048
NUM_SAMPLES = 5  # Number of random images to test

In [ ]:
def load_annotations(annotations_path):
    with open(annotations_path, "r") as f:
        return json.load(f)

def map_filename_to_caption(annotations):
    filename_to_id = {img["file_name"]: img["id"] for img in annotations["images"]}
    id_to_captions = {}
    for ann in annotations["annotations"]:
        img_id = ann["image_id"]
        id_to_captions.setdefault(img_id, []).append(ann["caption"])
    return filename_to_id, id_to_captions

print("Loading annotations...")
annotations = load_annotations(ANNOTATIONS_PATH)
filename_to_id, id_to_captions = map_filename_to_caption(annotations)
print("Annotations mapped successfully!")

In [ ]:
#model loading
print(f"Loading {MODEL_NAME}...")

if MODEL_NAME == "allenai/Molmo2-4B":
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME, trust_remote_code=True, dtype="auto", device_map="auto"
    )
elif MODEL_NAME == "google/gemma-4-E4B-it":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, dtype="auto", device_map="auto"
    )
else:
    raise ValueError(f"Unsupported model: {MODEL_NAME}")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME, trust_remote_code=True, dtype="auto", device_map="auto"
)

print("Model and processor loaded successfully.")

In [ ]:
#select a subset of images, generate captions

# 1. Select random subset
all_images = [f for f in os.listdir(DATASET_PATH) if f.endswith('.jpg')]
subset_images = random.sample(all_images, NUM_SAMPLES)

results_data = []

print(f"Generating captions for {NUM_SAMPLES} images using prompt: '{PROMPT}'...")

for file in subset_images:
    img_path = os.path.join(DATASET_PATH, file)
    image = Image.open(img_path).convert("RGB")
    
    # 2. Generate Text
    if MODEL_NAME == "allenai/Molmo2-4B":
        messages = [{"role": "user", "content": [{"type": "text", "text": PROMPT}, {"type": "image", "image": image}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)
            
        generated_tokens = generated_ids[0, inputs['input_ids'].size(1):]
        generated_text = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
    elif MODEL_NAME == "google/gemma-4-E4B-it":
        messages = [{"role": "user", "content": [{"type": "image", "url": image}, {"type": "text", "text": PROMPT}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
        input_len = inputs["input_ids"].shape[-1]
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)
            
        # skip_special_tokens=True ensures <eos> tags don't end up in the string
        generated_text = processor.decode(generated_ids[0][input_len:], skip_special_tokens=True)

    # 3. Clean and Store
    generated_text = generated_text.replace("<turn|>", "").strip()
    img_id = filename_to_id.get(file)
    gt_captions = id_to_captions.get(img_id, [])
    
    results_data.append({
        "file": file,
        "image_obj": image,
        "img_id": str(img_id), # Metrics require string IDs
        "generated": generated_text,
        "ground_truths": gt_captions
    })
    

print("Generation complete!")

In [ ]:
#visualize results before metrics 

for res in results_data:
    plt.figure(figsize=(5, 3))
    plt.imshow(res["image_obj"])
    plt.axis('off')
    plt.show()
    
    print(f"File: {res['file']}")
    print(f"Model Generated: {res['generated']}")
    print("Ground Truths:")
    for i, gt in enumerate(res["ground_truths"]):
        print(f"  - {gt}")
    print("-" * 60)

In [ ]:
#compute stats

# Format for pycocoevalcap: { id: ["caption"] }
gts = {res["img_id"]: res["ground_truths"] for res in results_data}
res_dict = {res["img_id"]: [res["generated"]] for res in results_data}

print("Computing CIDEr...")
cider_scorer = Cider()
cider_avg, cider_scores = cider_scorer.compute_score(gts, res_dict)
print(f"Subset Average CIDEr: {cider_avg:.4f}")

print("\nComputing SPICE...")
spice_scorer = Spice()
spice_avg, spice_scores = spice_scorer.compute_score(gts, res_dict)
print(f"Subset Average SPICE: {spice_avg:.4f}")

In [ ]:
#save prompt!

output_filename = "prompt_experiment_summary.txt"
example = results_data[0] # Grab the first item as an example

summary_text = f"""--- Prompt Experiment Summary ---
Model: {MODEL_NAME}
Prompt: {PROMPT}
Number of Samples: {NUM_SAMPLES}

--- Subset Metrics ---
CIDEr: {cider_avg:.4f}
SPICE: {spice_avg:.4f}

--- Example ---
Image File: {example['file']}
Model Generated: {example['generated']}
Ground Truths: 
{chr(10).join(['  - ' + gt for gt in example['ground_truths']])}
"""

with open(output_filename, "a") as f: # Using "a" to append multiple tests to the same file
    f.write(summary_text + "\n" + "="*50 + "\n\n")

print(f"Summary appended to {output_filename}")